In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
tasks=(
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "qm9_homo",
    "qm9_lumo",
    "qm9_homo_lumo_gap"
)

path_template = "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{}_{}_0219"
train_datasets = {}
for task in tasks:
    path = path_template.format("train", task)
    train_datasets[task] = datasets.load_from_disk(path)

# concat train_datasets
list_train_datasets = []
for task in tasks:
    list_train_datasets.append(train_datasets[task])

concat_train_datasets = datasets.concatenate_datasets(list_train_datasets)
concat_train_datasets.save_to_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_ablation-pp_0508')



Saving the dataset (4/4 shards): 100%|██████████| 423968/423968 [00:38<00:00, 11016.46 examples/s]


In [11]:
test_datasets = {}
for task in tasks:
    path = path_template.format("test", task)
    test_datasets[task] = datasets.load_from_disk(path)

# concat test_datasets
list_test_datasets = []
for task in tasks:
    list_test_datasets.append(test_datasets[task])

concat_test_datasets = datasets.concatenate_datasets(list_test_datasets)
concat_train_datasets.save_to_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ablation-pp_0508')

Saving the dataset (4/4 shards): 100%|██████████| 423968/423968 [00:18<00:00, 22329.66 examples/s] 


In [8]:
concat_test_datasets

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 9979
})

In [12]:
list(set(concat_train_datasets['task']))

['qm9_homo',
 'qm9_homo_lumo_gap',
 'qm9_lumo',
 'smol-property_prediction-clintox',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-esol',
 'smol-property_prediction-hiv',
 'smol-property_prediction-sider',
 'bace',
 'smol-property_prediction-lipo']

In [9]:
list(set(concat_test_datasets['task']))

['qm9_homo',
 'qm9_homo_lumo_gap',
 'qm9_lumo',
 'smol-property_prediction-clintox',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-esol',
 'smol-property_prediction-hiv',
 'smol-property_prediction-sider',
 'bace',
 'smol-property_prediction-lipo']

In [3]:
train_datasets

{'bace': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 1210
 }),
 'smol-property_prediction-bbbp': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 1569
 }),
 'smol-property_prediction-clintox': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 1144
 }),
 'smol-property_prediction-hiv': Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
     num_rows: 32864
 }),
 'smol-property_prediction-sider': Dataset({
     features